## Download text file

In [1]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !wget https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/Common-Credentials/10k-most-common.txt -O 10k-most-common.txt

--2022-08-14 15:30:53--  https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/Common-Credentials/10k-most-common.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 73017 (71K) [text/plain]
Saving to: ‘10k-most-common.txt’

10k-most-common.txt 100%[===================>]  71.31K  --.-KB/s    in 0.01s   

2022-08-14 15:30:53 (5.50 MB/s) - ‘10k-most-common.txt’ saved [73017/73017]



## Example code of 'Hash'

In [3]:
import hashlib

m=hashlib.sha1(b"Chulalongkorn").hexdigest()
m2=hashlib.sha1(b"Chulalongkorn University").hexdigest()
print(m)
print(m2)
m=hashlib.md5(b"Chulalongkorn").hexdigest()
m2=hashlib.md5(b"Chulalongkorn University").hexdigest()
print(m)
print(m2)

ca8a68498ae67cd14c15f5ebf043633224005759
a16c5b03cf3aca5c2f20169b4caa909d5c2f07ad
46fa3b56c660faff420190c18c98a56b
cc3fed293eb73ca7d3597a31259df950


## Implement Word Transformation Function

In [54]:
pair_num_lett = {'o':'0', 'l':'1', 'i': '1', 'O':'0', 'L':'1', 'I': '1'}

def get_all_pos_words(string_word):
  string_word = str(string_word)
  last_uppercase_word = string_word.upper()
  for key, val in pair_num_lett.items():
    last_uppercase_word = last_uppercase_word.replace(key, val)
  all_word_list = [string_word]
  last_word_list = [string_word]
  while last_uppercase_word != last_word_list[-1]:
    new_word_list = set()
    for word in last_word_list:
      for idx in range(len(word)):
        letter = word[idx]
        if letter.islower():
          copy_word = word
          copy_word = copy_word[:idx] + letter.upper() + copy_word[idx+1:]
          new_word_list.add(copy_word)
        if letter in pair_num_lett:
          copy_word = word
          copy_word = copy_word[:idx] + pair_num_lett[letter] + copy_word[idx+1:]
          new_word_list.add(copy_word)
    last_word_list = list(new_word_list)
    all_word_list.extend(last_word_list)
  return all_word_list

## Read '10k-most-common.txt' with Pandas

In [55]:
import time
import pandas as pd

common_words_df = pd.read_csv('/content/10k-most-common.txt', header=None, names=['word'])

## Transform words to all possible cases & Save to txt file

In [56]:
start_time = time.time()

common_words_df['word_list'] = common_words_df["word"].apply(get_all_pos_words)
nest_ls = common_words_df.word_list.values.tolist()
with open('/content/tf_words.txt', "w", encoding="UTF8", newline="") as file:
    for sub_ls in nest_ls:
      for word in sub_ls:
        file.write(word + '\n')

elapsed_time = time.time() - start_time

print(f"elapsed_time of 'Word Transformation' : {elapsed_time}")

elapsed_time of 'Word Transformation' : 33.59225249290466


## Hash passwords & Save to csv file

In [49]:
start_time2 = time.time()

def hash_word(word):
  return hashlib.sha1(str(word).encode()).hexdigest()

password_df = pd.read_csv('/content/tf_words.txt', header=None, names=['password'])
password_df['hashed_password'] = password_df['password'].apply(hash_word)
password_df.to_csv(f'hash_tables.csv', index=False)

elapsed_time2 = time.time() - start_time2

print(f"elapsed_time of 'Hashing passwords' : {elapsed_time2}")

elapsed_time of 'Hashing passwords' : 22.67829704284668


In [57]:
original_value = 'd54cc1fe76f5186380a0939d2fc1723c44e8a5f7'
original_password = password_df[password_df['hashed_password'] == original_value]['password'].values[0]

print(f"original_password is : {original_password}")

original_password is : ThaiLanD
